# Agents digital loop — RTL-to-GDS on a 4-bit counter (GF180MCU)

Walks the digital loop: wrap a minimal LibreLane config with `GenericDesign`,
hand it to `ProjectManager` (the master ADK LlmAgent), and watch it plan the
flow through `verification_engineer` → `synthesis_engineer` → `physical_designer`
→ `signoff_checker`.

Uses the same counter RTL + `config.yaml` as
`../../rtl2gds-gf180-docker/demo/rtl2gds_counter.ipynb` so both tutorials agree.
Companion to the slide deck at `../claude_design_slides/`.

**Default mode is DRY-RUN.** Opening this notebook is a safe no-op. Flip
`RUN_REAL=True` only after the dry-run cell completes cleanly — a real run
takes ~10-15 min and burns LLM tokens.

**Prerequisites**

- You cloned `eda-agents` — this notebook lives at `tutorials/agents-analog-digital/demo/`.
- Activated venv (Python 3.9+).
- Docker Engine available; real-run cell pulls `hpretl/iic-osic-tools:next`.
- Either `$OPENROUTER_API_KEY` / `$GOOGLE_API_KEY` / `$ZAI_API_KEY` for ADK
  backend, **or** Claude Code CLI on PATH for `cc_cli` backend.

## Step 0 — Editable install from the repo root

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

RUN_PIP_INSTALL  = False   # flip once per venv
RUN_DRY_PM       = True    # safe, fast — default on
RUN_REAL         = False   # flip only after dry-run passes
RUN_DANGEROUSLY  = False   # gate for cc_cli --dangerously-skip-permissions; also
                           # requires EDA_AGENTS_ALLOW_DANGEROUS=1 in the env

BACKEND = "cc_cli"                           # "adk" | "cc_cli"

# Backend-aware default model. cc_cli forwards --model to `claude --print`,
# which only accepts Anthropic IDs; passing google/gemini there returns API 404.
# adk uses LiteLLM and accepts provider-prefixed IDs.
_DEFAULT_MODEL = "claude-sonnet-4-6" if BACKEND == "cc_cli" else "google/gemini-3-flash-preview"
LLM_MODEL = os.environ.get("EDA_AGENTS_MODEL", _DEFAULT_MODEL)
MAX_BUDGET_USD = 1.00                        # meaningful for cc_cli only

REPO_ROOT = Path.cwd().resolve().parents[2]
WORK_DIR  = Path.cwd() / "rtl2gds_counter_results"
PROJ_DIR  = WORK_DIR / "counter_project"

print(f"Repo root : {REPO_ROOT}")
print(f"Python    : {sys.executable}")
print(f"venv      : {os.environ.get('VIRTUAL_ENV', 'UNSET (activate a venv first)')}")

if RUN_PIP_INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
else:
    print("[rehearse] RUN_PIP_INSTALL=False — skipping install.")

## Step 1 — Docker, image, CLI, API-key check

In [ ]:
import shutil

docker = shutil.which("docker")
claude = shutil.which("claude")
print(f"docker on PATH         : {docker or 'MISSING'}")
if docker:
    try:
        out = subprocess.run(
            ["docker", "images", "-q", "hpretl/iic-osic-tools:next"],
            capture_output=True, text=True, timeout=10,
        )
        img = out.stdout.strip()
        print(f"hpretl image present   : {'yes (' + img[:12] + ')' if img else 'no -- docker pull required'}")
    except subprocess.SubprocessError as exc:
        print(f"docker images query failed: {exc}")
print(f"claude CLI on PATH     : {claude or 'MISSING (needed for cc_cli backend)'}")
for key in ("OPENROUTER_API_KEY", "GOOGLE_API_KEY", "ZAI_API_KEY"):
    print(f"{key:<22}: {'set' if os.environ.get(key) else 'unset'}")

## Step 2 — Stage `counter.v` + `config.yaml` on the host

Inlined so the notebook is self-contained. Identical to the sibling
`rtl2gds_counter.ipynb` so both tutorials use the same counter.

In [ ]:
COUNTER_V = '''// 4-bit synchronous up-counter with active-high reset.
module counter (
    input  wire       clk,
    input  wire       rst,
    output wire [3:0] q
);
    reg [3:0] cnt;
    always @(posedge clk) begin
        if (rst) cnt <= 4'b0;
        else     cnt <= cnt + 4'b1;
    end
    assign q = cnt;
endmodule
'''

CONFIG_YAML = '''# GF180MCU LibreLane config - minimal 4-bit counter walk-through.
meta:
  version: 3
  flow: Classic
  substituting_steps:
    Magic.StreamOut: null
    KLayout.XOR: null
    KLayout.DRC: null            # gf180mcuD ships no KLAYOUT_DRC_RUNSET; the step crashes
                                 # with "Unable to open file: .../None" when librelane is
                                 # invoked without --manual-pdk (the typical agent path).
                                 # Magic DRC at 0 is the authoritative DRC for this PDK.

RUN_MAGIC_STREAMOUT: false
RUN_KLAYOUT_XOR: false
RUN_KLAYOUT_DRC: false

DESIGN_NAME: counter
VERILOG_FILES:
  - dir::counter.v
CLOCK_PORT: clk
CLOCK_PERIOD: 50

FP_SIZING: absolute
DIE_AREA: [0.0, 0.0, 300.0, 300.0]

VDD_NETS:
  - VDD
GND_NETS:
  - VSS

PRIMARY_GDSII_STREAMOUT_TOOL: klayout

DIODE_ON_PORTS: in
RT_MAX_LAYER: Metal4
PDN_MULTILAYER: false
PDN_VWIDTH: 5
PDN_HWIDTH: 5
PDN_VSPACING: 1
PDN_HSPACING: 1
PDN_VOFFSET: 10
PDN_HOFFSET: 10
'''

PROJ_DIR.mkdir(parents=True, exist_ok=True)
(PROJ_DIR / "counter.v").write_text(COUNTER_V)
(PROJ_DIR / "config.yaml").write_text(CONFIG_YAML)
for p in PROJ_DIR.iterdir():
    print(f"wrote {p}")

## Step 3 — Construct `ProjectManager` and dry-run

`GenericDesign` turns any LibreLane config into a 13-method `DigitalDesign` the
agents understand — no per-project Python class required. `ProjectManager`
wraps the master LlmAgent; its `.run(work_dir, dry_run=True)` builds the prompt
and lists sub-agents without contacting the LLM or Docker. Safe to re-run.

`BACKEND="cc_cli"` delegates to `claude --print`; `BACKEND="adk"` uses the
Python ADK runner.

In [ ]:
import asyncio
from eda_agents.core.designs.generic import GenericDesign
from eda_agents.agents.digital_adk_agents import ProjectManager

async def _dry():
    design = GenericDesign(
        config_path=str(PROJ_DIR / "config.yaml"),
        pdk_root=os.environ.get("PDK_ROOT") or None,
        pdk_config="gf180mcu",
    )
    pm = ProjectManager(
        design=design,
        model=LLM_MODEL,
        backend=BACKEND,
        max_budget_usd=MAX_BUDGET_USD,
        allow_dangerous=RUN_DANGEROUSLY,
    )
    result = await pm.run(WORK_DIR, dry_run=True)
    return design, result

if RUN_DRY_PM:
    design, result = asyncio.get_event_loop().run_until_complete(_dry())
    print(f"design          : {design.project_name()}")
    print(f"specs           : {design.specs_description()}")
    print(f"FoM             : {design.fom_description()}")
    print(f"backend         : {BACKEND}")
    if BACKEND == "cc_cli":
        print(f"prompt length   : {result.get('prompt_length', 0)} chars")
    else:
        subs = result.get("sub_agent_names") or result.get("sub_agents") or []
        print(f"master          : {result.get('master_agent', 'N/A')}")
        print(f"sub-agents      : {', '.join(str(s) for s in subs)}")
else:
    print("[rehearse] RUN_DRY_PM=False — skipping.")

## Step 4 — Full RTL-to-GDS run (gated, expensive)

Only flip `RUN_REAL=True` once dry-run printed a prompt and the Docker +
image + API-key / CLI checks above all looked green.

Expected wall time: ~10-15 minutes for this counter. The agent walks
lint → cocotb sim → Yosys synth → PnR → signoff and writes
`rtl2gds_results.json` under the work dir.

If the run fails: open `rtl2gds_results.json` and grep for `error`. The
master agent's `agent_output` field is the full trace — read it, don't
guess.

In [ ]:
async def _real():
    if BACKEND == "cc_cli" and not (RUN_DANGEROUSLY and os.environ.get("EDA_AGENTS_ALLOW_DANGEROUS") == "1"):
        raise RuntimeError(
            "cc_cli backend in non-interactive mode needs RUN_DANGEROUSLY=True "
            "and EDA_AGENTS_ALLOW_DANGEROUS=1 in the env. Or switch BACKEND='adk'."
        )
    design = GenericDesign(
        config_path=str(PROJ_DIR / "config.yaml"),
        pdk_root=os.environ.get("PDK_ROOT") or None,
        pdk_config="gf180mcu",
    )
    pm = ProjectManager(
        design=design,
        model=LLM_MODEL,
        backend=BACKEND,
        max_budget_usd=MAX_BUDGET_USD,
        allow_dangerous=RUN_DANGEROUSLY,
    )
    return await pm.run(WORK_DIR)

if RUN_REAL:
    import json
    res = asyncio.get_event_loop().run_until_complete(_real())
    slim = {k: v for k, v in res.items() if k not in ("agent_output", "prompt")}
    print(json.dumps(slim, indent=2, default=str))
else:
    print("[rehearse] RUN_REAL=False — skipping.")
    print(f"When enabled: artifacts land under {WORK_DIR}")
    print("Expected wall time: ~10-15 min for this counter.")

## Step 5 — Read the metrics

LibreLane drops `metrics.csv` per run under `runs/<timestamp>/final/`. The
ProjectManager trace is in `rtl2gds_results.json` next to the work dir.

In [ ]:
results = WORK_DIR / "rtl2gds_results.json"
if results.exists():
    print(f"--- {results} ---")
    print(results.read_text()[:2000])
else:
    print(f"{results} not yet written; run step 4 first.")

runs = PROJ_DIR / "runs"
if runs.is_dir():
    latest = max(runs.iterdir(), key=lambda p: p.stat().st_mtime, default=None)
    if latest:
        metrics = latest / "final" / "metrics.csv"
        if metrics.exists():
            print(f"\n--- {metrics} (first rows) ---")
            print(metrics.read_text()[:1500])

## Cleanup

```bash
rm -rf rtl2gds_counter_results/
# (optional) reclaim container workspace:
docker rm -f gf180 2>/dev/null || true
```

## Next

- Analog-loop companion: `./agents_miller_ota.ipynb`.
- Slide deck: `../claude_design_slides/AI Agents for Analog and Digital IC Design.html`.
- For the sibling Docker-driven standalone flow (no Python agents, just the
  bare LibreLane pipeline): `../../rtl2gds-gf180-docker/demo/rtl2gds_counter.ipynb`.
- Source of truth for agent + tool allowlists:
  `src/eda_agents/agents/digital_adk_agents.py`.